# Corpus Bootstrap

Your `local_corpus.json` (1,021 FastAPI chunks) is **already built and compatible**  
with `VersionControlRAG.load_local_corpus()`. You just need to:

1. Put `local_corpus.json` and `processed_state.json` next to this notebook
2. Run **Path A** — restores BM25 from disk (fast, no LLM needed)
3. Run **Path B** — re-upserts to Pinecone (only needed once, or after index reset)

```
your_project/
├── local_corpus.json          ← from your generated files
├── processed_state.json       ← from your generated files  
├── rag_pipeline.py
├── repoProcessor.py
└── this_notebook.ipynb
```

In [ ]:
import rag_pipeline, importlib
importlib.reload(rag_pipeline)
from rag_pipeline import VersionControlRAG
from pathlib import Path

CORPUS_FILE = "local_corpus.json"      # ← make sure this file is in the same folder
STATE_FILE  = "processed_state.json"
PINECONE_KEY = "YOUR_PINECONE_API_KEY" # ← your key

# Verify files exist before doing anything
for f in [CORPUS_FILE, STATE_FILE]:
    status = "✅" if Path(f).exists() else "❌ NOT FOUND — copy this file here first"
    print(f"{status}  {f}  ", end="")
    if Path(f).exists():
        import json
        with open(f) as fh:
            data = json.load(fh)
        print(f"({len(data)} items)")
    else:
        print()

## Path A — Fast restore (BM25 only, no Pinecone upsert)

Use this on every kernel restart. Takes ~5 seconds.  
Skips Pinecone re-ingestion — assumes vectors are already there from a previous session.

In [ ]:
# Load the RAG system WITHOUT re-ingesting (base model only, no adapter needed for retrieval)
rag = VersionControlRAG(
    pinecone_key = PINECONE_KEY,
    model_path   = "Qwen/Qwen2.5-Coder-7B",
    adapter_path = "Ivan17Ji/qwen-lora-250",
)

# This call does everything:
#   1. Reads local_corpus.json into self.local_corpus
#   2. Rebuilds self._corpus_ids set (duplicate detection)
#   3. Rebuilds BM25 index from the corpus
rag.load_local_corpus(CORPUS_FILE)

print(f"\n✅ Corpus loaded: {len(rag.local_corpus)} docs")
print(f"✅ BM25 ready:    {rag.bm25_model is not None}")

# Quick sanity: try a retrieval
test = rag.retrieve_complex("How to define an APIRouter?", target_version="3.10", top_k=1, mode="advanced")
print(f"✅ Test retrieval: {len(test)} result(s)")
if test:
    print("  Top result preview:", test[0][:80].strip())

## Path B — Full re-ingest to Pinecone

Run this **only if**:
- Your Pinecone index was reset or deleted
- This is your first time setting up the index
- Retrieval returns 0 results even after Path A

Takes a few minutes (1,021 vectors to upsert in batches of 100).

In [ ]:
# ── Check if Pinecone index already has vectors ───────────────────────────────
stats = rag.index.describe_index_stats()
total_vectors = stats.get("total_vector_count", 0)
print(f"Pinecone index stats: {total_vectors} vectors currently stored")

if total_vectors >= len(rag.local_corpus):
    print("✅ Index looks full — Path A is sufficient, skip Path B")
elif total_vectors > 0:
    print(f"⚠  Partial index ({total_vectors}/{len(rag.local_corpus)}) — consider running Path B")
else:
    print("❌ Index is empty — run the cell below to re-ingest")

In [ ]:
# ── Re-upsert all corpus vectors to Pinecone ─────────────────────────────────
# Only run this if the cell above shows the index is empty or partial.

import tqdm, json

with open(CORPUS_FILE) as f:
    corpus_items = json.load(f)

print(f"Re-ingesting {len(corpus_items)} items into Pinecone...")

# Reset the buffer and corpus_ids so ingest_data accepts every item
rag._upsert_buffer = []
rag._corpus_ids    = set()   # temporarily clear so nothing is skipped

for item in tqdm.tqdm(corpus_items, desc="Upserting"):
    rag.ingest_data(
        doc_id      = item["id"],
        code_content= item["content"],
        min_version = str(item["metadata"]["min_version"]),   # ingest_data calls _parse_version
        summary     = item["metadata"]["summary"],
    )

# Flush any remaining items in the buffer
rag.flush_upsert_buffer()

# Restore corpus_ids from the loaded corpus
rag._corpus_ids = {doc["id"] for doc in rag.local_corpus}

print(f"✅ Done. Upserted {len(corpus_items)} vectors.")

# Verify
stats = rag.index.describe_index_stats()
print(f"Pinecone now has: {stats.get('total_vector_count', 0)} vectors")

## Path C — Fresh ingest from the FastAPI repo

Only needed if you want to re-build the corpus from scratch using your QLoRA to generate summaries.  
This takes a **long time** (runs `get_qwen_metadata` on every function).  
Use Path A or B unless you specifically want to rebuild.

In [ ]:
# Clone FastAPI if not already present
import os
if not os.path.isdir("./fastapi"):
    print("Cloning FastAPI repo...")
    os.system("git clone --depth=1 https://github.com/tiangolo/fastapi")
else:
    print("✅ ./fastapi already exists")

In [ ]:
# This will take a long time — runs QLoRA on every function to generate metadata
# Only run if you want a fresh corpus, not just to restore state

import repoProcessor, importlib
importlib.reload(repoProcessor)
from repoProcessor import RepoProcessor

processor = RepoProcessor(rag, state_file=STATE_FILE)
processor.process_repository("./fastapi", "FastAPI-Core")

rag.save_local_corpus(CORPUS_FILE)  # save result to disk
print("✅ Fresh corpus built and saved")

## Verify everything is ready
Run this before your eval notebook to confirm the system is fully operational.

In [ ]:
print("=" * 50)
print("System readiness check")
print("=" * 50)

checks = [
    ("Corpus loaded",  len(rag.local_corpus) > 0,      f"{len(rag.local_corpus)} docs"),
    ("BM25 ready",     rag.bm25_model is not None,     "index built"),
    ("Model loaded",   rag.model is not None,           "Qwen ready"),
    ("Tokenizer",      rag.tokenizer is not None,       "ready"),
]

# Pinecone check
try:
    stats = rag.index.describe_index_stats()
    n_vec = stats.get("total_vector_count", 0)
    checks.append(("Pinecone index", n_vec > 0, f"{n_vec} vectors"))
except Exception as e:
    checks.append(("Pinecone index", False, str(e)))

# Retrieval smoke test
try:
    res_base = rag.retrieve_complex("FastAPI route decorator", target_version="3.10", top_k=3, mode="baseline")
    res_adv  = rag.retrieve_complex("FastAPI route decorator", target_version="3.10", top_k=3, mode="advanced")
    checks.append(("Baseline retrieval", len(res_base) > 0, f"{len(res_base)} results"))
    checks.append(("Advanced retrieval", len(res_adv)  > 0, f"{len(res_adv)} results"))
except Exception as e:
    checks.append(("Retrieval", False, str(e)))

all_ok = True
for name, passed, detail in checks:
    icon = "✅" if passed else "❌"
    print(f"  {icon}  {name:<22} {detail}")
    if not passed:
        all_ok = False

print()
if all_ok:
    print("✅ All checks passed — open ragas_rag_pipeline_eval.ipynb to run the full eval")
else:
    print("❌ Fix the failing checks above, then re-run this cell")